# Predicting Unusual Household and Housing Profiles Related to Social Tenancy Risk Using Machine Learning
Ahmed Zaus Zahid, Afnan Ali, Aminath Iuzaaz Ismail



### Loading the data
The AHS dataset is a massive dataset. However, most of the data is not related to our specific usecase and we will not be needing them. Before we can train our models on the data, we need to identify which columns we need and treat the data which requires cleaning. 

In [5]:
import pandas as pd

# the columns we need from the dataset
ahs_cols = [
    "CONTROL", "TENURE", "WEIGHT",
    "HUDSUB", "RENTSUB", "RENTCNTRL",
    "HINCP", "FINCP", "PERPOVLVL", "FS",
    "NUMPEOPLE", "BEDROOMS", "TOTROOMS",
    "NUMADULTS", "NUMELDERS", "NUMYNGKIDS", "NUMOLDKIDS", "UNITSIZE",
    "RENT", "TOTHCAMT", "UTILAMT",
    "HIAFFORD", "HIBEHINDFRQ", "HIHALF", "HINUMOVE",
    "HIEVICNOTE", "HIEVICTHT", "HIEVICLK", "HINOWHERE",
    "MGRONSITE",
    "NUMNONREL", "NUMSECFAM", "NUMSUBFAM",
    "OCCJANUR", "OCCFEBRU", "OCCMARCH", "OCCAPRIL", "OCCMAY", "OCCJUNE",
    "OCCJULY", "OCCAUGUST", "OCCSEPTEM", "OCCOCTOB", "OCCNOVEM", "OCCDECEM",
    "OCCYRRND", "MONLSTOCC", "VACMONTHS",
]

df = pd.read_csv(
    "household.csv",             # AHS 2023 national PUF file
    quotechar="'",               # strips the single quotes around every value
    usecols=lambda c: c in ahs_cols,   # only load these columns
)


print(df.shape)
print(df.head())

(55669, 48)
    CONTROL  TOTROOMS  PERPOVLVL  RENT  TENURE  HIEVICLK  HIAFFORD  HIHALF  \
0  11000002         6         -6  1600      -6        -6        -6      -6   
1  11000003         4        199   840       2         4         4       2   
2  11000005         7        501    -6       1        -6        -6      -6   
3  11000006         5        232    -6       1        -6        -6      -6   
4  11000008         3        231   800       2         2         2       1   

   OCCJANUR  OCCFEBRU  ...  HIBEHINDFRQ  HINOWHERE  HIEVICTHT  HIEVICNOTE  \
0         1         1  ...           -6         -6         -6          -6   
1        -6        -6  ...            6          0          2          -6   
2        -6        -6  ...           -6          0         -6          -6   
3        -6        -6  ...           -6         -6         -6          -6   
4        -6        -6  ...            2          0          1           1   

   HINUMOVE  MGRONSITE  HUDSUB   HINCP   FINCP  TOTHCAMT

Running the cell, we can already see that our data needs cleaning even though the printed section is only a small slice of the full data. The AHS dataset uses -6 and -9 to represent Null values.

### Analysing the data for cleaning

In [6]:
r = df[df["TENURE"] == 2].replace([-6, -9], pd.NA)   # renters only, codes -> NaN
missing = r.isna().mean().sort_values(ascending=False)
print((missing * 100).round(1))
print("renter rows:", len(r))

OCCAUGUST      100.0
OCCJULY        100.0
OCCJUNE        100.0
OCCMAY         100.0
OCCAPRIL       100.0
OCCMARCH       100.0
OCCFEBRU       100.0
OCCJANUR       100.0
OCCDECEM       100.0
VACMONTHS      100.0
MONLSTOCC      100.0
OCCYRRND       100.0
OCCOCTOB       100.0
OCCNOVEM       100.0
OCCSEPTEM      100.0
HIEVICNOTE      98.1
RENTCNTRL       87.6
HIHALF          52.6
HIEVICLK        51.7
HIAFFORD        51.7
HIBEHINDFRQ     51.6
HINOWHERE       51.6
HIEVICTHT       51.5
HINUMOVE        50.7
MGRONSITE       31.0
FS              30.0
UNITSIZE        18.0
RENTSUB          2.0
TENURE           0.0
PERPOVLVL        0.0
TOTROOMS         0.0
CONTROL          0.0
RENT             0.0
NUMSUBFAM        0.0
NUMOLDKIDS       0.0
NUMYNGKIDS       0.0
NUMNONREL        0.0
NUMADULTS        0.0
NUMELDERS        0.0
WEIGHT           0.0
NUMSECFAM        0.0
UTILAMT          0.0
NUMPEOPLE        0.0
BEDROOMS         0.0
HUDSUB           0.0
HINCP            0.0
FINCP            0.0
TOTHCAMT     

Filtering by TENURE == 2 (rented) and checking for missing data values per column. We have to drop all the columns without enough data. Columns that have 50% missing data can be considered, however we will ignore those for now and come back to it if the model requires more signals.

### Updated columns after removing columns with missing data

In [7]:
KEEP_COLS = [
    "CONTROL", "TENURE", "WEIGHT",
    "HUDSUB", "RENTSUB", "RENTCNTRL", "HHMOVE",
    "HINCP", "FINCP", "PERPOVLVL", "FS",
    "NUMPEOPLE", "BEDROOMS", "TOTROOMS",
    "NUMADULTS", "NUMELDERS", "NUMYNGKIDS", "NUMOLDKIDS", "UNITSIZE",
    "RENT", "TOTHCAMT", "UTILAMT",
    "MGRONSITE",
    "NUMNONREL", "NUMSECFAM", "NUMSUBFAM",
]

DROP_ALWAYS = (
    ["OCCJANUR", "OCCFEBRU", "OCCMARCH", "OCCAPRIL", "OCCMAY", "OCCJUNE",
     "OCCJULY", "OCCAUGUST", "OCCSEPTEM", "OCCOCTOB", "OCCNOVEM", "OCCDECEM",
     "OCCYRRND", "VACMONTHS", "MONLSTOCC"]        # 100% empty on renters
    + ["HIEVICNOTE"]                               # 98% empty (skip pattern)
    + ["HIAFFORD", "HIBEHINDFRQ", "HIHALF", "HINUMOVE",
       "HIEVICLK", "HIEVICTHT", "HINOWHERE"]       # ~50% (split-sample)
)

### Loading the data again

In [8]:
df = pd.read_csv(
    "household.csv",
    quotechar="'",
    usecols=lambda c: c in KEEP_COLS,
)
print(df.shape)
df.head()

(55669, 26)


,CONTROL,TOTROOMS,PERPOVLVL,RENT,TENURE,RENTCNTRL,RENTSUB,WEIGHT,HHMOVE,NUMELDERS,...,NUMPEOPLE,UNITSIZE,BEDROOMS,UTILAMT,FS,MGRONSITE,HUDSUB,HINCP,FINCP,TOTHCAMT
0,11000002,6,-6,1600,-6,-6,8,813.890194,-6,-6,...,-6,7,4,0,-6,-6,-6,-6,-6,-6
1,11000003,4,199,840,2,-6,8,581.103231,2023,0,...,3,-9,2,240,2,-6,3,48000,48000,1093
2,11000005,7,501,-6,1,-6,-6,7335.965001,1995,2,...,2,6,4,260,-6,-6,-6,292500,292500,810
3,11000006,5,232,-6,1,-6,-6,6562.865941,2019,0,...,3,4,3,270,2,-6,-6,56000,56000,489
4,11000008,3,231,800,2,-6,8,1490.800600,2019,0,...,1,-9,1,4,-6,4,3,36000,36000,845


### Replacing -6 and -9 with NA

In [9]:
df = df.replace([-6, -9], pd.NA)
df = df[df["TENURE"] == 2].copy()
print(f"{len(df)} renter rows")

19735 renter rows


### Dropping columns

In [10]:
df = df.drop(columns=[c for c in DROP_ALWAYS if c in df.columns])
print(f"{df.shape[1]} columns kept")

26 columns kept


### Recoding NA flags to 0

In [11]:
for col in ["RENTCNTRL", "MGRONSITE"]:
    df[col] = df[col].fillna(0)

In [12]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df.dtypes

CONTROL         int64
TOTROOMS        int64
PERPOVLVL       int64
RENT            int64
TENURE          int64
RENTCNTRL       int64
RENTSUB       float64
WEIGHT        float64
HHMOVE          int64
NUMELDERS       int64
NUMADULTS       int64
NUMNONREL       int64
NUMYNGKIDS      int64
NUMOLDKIDS      int64
NUMSUBFAM       int64
NUMSECFAM       int64
NUMPEOPLE       int64
UNITSIZE      float64
BEDROOMS        int64
UTILAMT         int64
FS            float64
MGRONSITE       int64
HUDSUB          int64
HINCP           int64
FINCP           int64
TOTHCAMT        int64
dtype: object

In [13]:
df["persons_per_room"] = df["NUMPEOPLE"] / df["TOTROOMS"]
df["is_crowded"] = (df["persons_per_room"] > 1).astype(int)
df["is_crowded"].value_counts()

is_crowded
0    19152
1      583
Name: count, dtype: int64

In [14]:
miss = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
miss[miss > 0]

FS          30.0
UNITSIZE    18.0
RENTSUB      2.0
dtype: float64

In [15]:
df = df.drop(columns=["UNITSIZE"])
df = df.drop(columns=["FS"])

In [16]:
# --- component flags (each maps to a real misuse indicator) ---

# 1. Subsidy/eligibility anomaly: subsidised but income high for need.
#    IMPORTANT: check HUDSUB's codes in the codebook first - it encodes
#    eligibility directly, so confirm which value means "subsidised".
df["flag_subsidy_anomaly"] = (
    (df["HUDSUB"] == 3) &          # <-- replace with the real "subsidised" code
    (df["PERPOVLVL"] > 200)        # income > 200% of poverty threshold
).astype(int)

# 2. Unrelated adults present (potential sub-tenants)
df["flag_unrelated"] = (df["NUMNONREL"] >= 2).astype(int)

# 3. Secondary / sub-family in one unit (subdivision into lets)
df["flag_secondary_family"] = (
    (df["NUMSECFAM"] >= 1) | (df["NUMSUBFAM"] >= 1)
).astype(int)

# 4. Overcrowding (already built)
df["flag_crowded"] = df["is_crowded"]

# 5. Recent move-in  (needs HHMOVE - add it to KEEP_COLS and reload)
df["flag_recent_move"] = (df["HHMOVE"] >= 2022).astype(int)

# --- weighted additive score ---
df["risk_score"] = (
    2 * df["flag_subsidy_anomaly"]
    + 2 * df["flag_unrelated"]
    + 2 * df["flag_secondary_family"]
    + 1 * df["flag_crowded"]
    + 1 * df["flag_recent_move"]
)

# --- cut into your four tiers ---
def to_tier(s):
    if s == 0:  return "Low Concern"
    if s <= 2:  return "Monitor"
    if s <= 4:  return "Investigate"
    return "Priority Action"

df["risk_tier"] = df["risk_score"].apply(to_tier)
df["risk_tier"].value_counts()

risk_tier
Low Concern        8394
Monitor            8050
Investigate        3046
Priority Action     245
Name: count, dtype: int64

In [25]:
df.sort_values("risk_score", ascending=False)

,CONTROL,TOTROOMS,PERPOVLVL,RENT,TENURE,RENTCNTRL,RENTSUB,WEIGHT,HHMOVE,NUMELDERS,...,TOTHCAMT,persons_per_room,is_crowded,flag_subsidy_anomaly,flag_unrelated,flag_secondary_family,flag_crowded,flag_recent_move,risk_score,risk_tier
8112,11014487,4,372,2400,2,2,8.0,5834.888385,2022,0,...,2683,1.500000,1,1,1,1,1,1,8,Priority Action
54525,11097674,5,230,900,2,0,8.0,580.286536,2022,0,...,1314,1.200000,1,1,1,1,1,1,8,Priority Action
8993,11016062,8,501,30,2,0,8.0,8334.652868,2023,0,...,30,0.500000,0,1,1,1,0,1,7,Priority Action
13120,11023633,5,414,1500,2,0,8.0,2152.731592,2022,0,...,1508,0.800000,0,1,1,1,0,1,7,Priority Action
13267,11023900,6,468,6100,2,0,8.0,1718.636104,2022,0,...,6220,0.666667,0,1,1,1,0,1,7,Priority Action
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55583,11099993,4,23,1900,2,0,8.0,1657.680118,2020,0,...,1940,0.250000,0,0,0,0,0,0,0,Low Concern
88,11000201,3,129,170,2,0,1.0,341.176577,2002,0,...,370,0.333333,0,0,0,0,0,0,0,Low Concern
87,11000199,6,1,70,2,0,2.0,1347.236543,2015,0,...,330,0.333333,0,0,0,0,0,0,0,Low Concern
86,11000198,4,125,760,2,0,5.0,370.473390,2019,0,...,850,0.750000,0,0,0,0,0,0,0,Low Concern
